<a href="https://colab.research.google.com/github/Saransh-Basu-01/50-Ml-Projects-to-understand-LLMs/blob/main/Project_34_QKV_Attention_Characterstics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

import requests

import torch
from transformers import AutoModelForCausalLM, GPT2Tokenizer
import torch.nn.functional as F

# Part 1: Model, tokens, QKV activations

## Task_1:
- Import GPT-2-large and its tokenizer. Switch the model to eval
mode. Define variables for the embeddings dimensionality and the
number of transformer layers.
Import the text for the Great Gatsby book from gutenberg.org.
Tokenize the book and find all instances of the token “ like” (note
the preceding space).
Create a batch of 114 × 51 for the 114 target token appear
ances and the 50 context-providing tokens before each target token.
Check whether any target tokens appear within the first 50 tokens of
the book, and if so, exclude them

In [2]:

# load GPT2 model and tokenizer
model = AutoModelForCausalLM.from_pretrained('gpt2-large')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

model.eval()

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.25GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1280)
    (wpe): Embedding(1024, 1280)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-35): 36 x GPT2Block(
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3840, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=1280)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=5120, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=5120)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1280,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=1280, out_features=50257, bias=False)
)

In [3]:

# variable for the number of embedding dimensions
n_emb = model.config.n_embd

# and the number of layers
n_layers =model.config.n_layer


In [13]:
# GreatGatsby
url = 'https://www.gutenberg.org/ebooks/64317.txt.utf-8'
text = requests.get(url).text
tokens = np.array( tokenizer.encode(text) )

target_word = ' like'

target_idx = np.where(tokens==tokenizer.encode(target_word))[0]
print(f'{len(target_idx)} instances of "{target_word}"')

0 instances of " like"


In [14]:
text

'<!DOCTYPE html> <html lang="en"> <head> <meta charset="utf-8"> <meta name="viewport" content="width=device-width, initial-scale=1"> <title>Temporarily Down for Maintenance</title> <style> body { margin: 0; font: 1.1em/1.6 Helvetica, Arial, sans-serif; color: #222; background: #f4f4f4; } main { max-width: 36em; margin: 15vh auto 0; padding: 2em; background: #fff; border-top: 6px solid #7BAFD4; } h1 { margin-top: 0; font-size: 1.6em; } </style> </head> <body> <main> <h1>Temporarily Down for Maintenance</h1> <p>Project Gutenberg is temporarily unavailable while we perform maintenance.</p> <p>Please check back soon. We apologize for the inconvenience.</p> <p>Follow Project Gutenberg on <a href="https://mastodon.social/@gutenberg_org">Mastodon</a>, <a href="https://bsky.app/profile/gutenberg.org">Bluesky</a>, or <a href="https://www.facebook.com/share/1A6Rcy9TLd/">Facebook</a>.</p> </main> </body> </html>'

In [15]:
# size of pre-target context (tokens)
contextwin = 50

# remove any target indices that are = contextwin]

print(f'Target count after filtering: {len(target_idx)}')

Target count after filtering: 0


In [16]:
# create a batch
batch = torch.zeros((len(target_idx),contextwin+1),dtype=torch.long)

for i in range(len(target_idx)):
  toks = tokens[target_idx[i]-contextwin:target_idx[i]+1]
  batch[i,:] = torch.tensor(toks)

batch.shape

torch.Size([0, 51])

## Task_2:
- Write a hook function to get attention activations from the
attn.c_attn output. Store the QKV matrix in a dictionary called
activations, with the key name including the layer number (e.g.,
attn_4_qkv). Implant the hook into all transformer layers.
Run a forward pass of the batch. Print out all the dictionary
keys and their shapes, and make sure you understand what the
tensor sizes correspond to.

In [ ]:

# Define a hook function to store QKV vectors
activations = {}

def implant_hook(layer_number):
  def hook(module,input,output):
    activations[f'attn_{layer_number}_qkv'] = output.detach()
  return hook


# surgery ;)
handles = []
for i in range(n_layers):
  h = model.transformer.h[i].attn.c_attn.register_forward_hook(implant_hook(i))
  handles.append(h)




In [ ]:
# push through the model
with torch.no_grad(): model(batch)

for k,i in activations.items():
  print(f'{k:>12} has shape {list(i.shape)}')

## Task_3:
- Visualize the activation matrices for the target token in layer index
6 as a heatmap as in Figure 6.12

In [ ]:

# concatenated activations from one layer
layeri = 6

wide_acts = activations[f'attn_{layeri}_qkv']

plt.figure(figsize=(10,3))
plt.imshow(wide_acts[:,-1,:],aspect='auto',vmin=-1,vmax=1,cmap='plasma')
plt.axvline(n_emb,linestyle='--',color='w')
plt.axvline(2*n_emb,linestyle='--',color='w')
plt.colorbar(pad=.01)

plt.gca().set(xticks=[],ylabel='Batch sequences',
              xlabel='Queries dimensions         |           Keys dimensions             |           Values dimensions')


plt.tight_layout()
plt.savefig('ch6_proj34_part1.png')
plt.show()

# Part 2: Activation characteristics

## Task_1:
- Split the activations matrix into Q, K, and V matrices. Confirm
that the sizes of the matrices match your expectations. For this task,
use only the Q vectors.
Calculate the three descriptive statistics listed above, for the
target (final token) and non-target (penultimate token). Make
sure you’re averaging the activations vectors; you should have one
statistical characteristic for each of the 114 sequences.
Average over the 114 values to create bar plots as in Figure
6.13. Interpret the differences between target and non-target, and
the differences between the different descriptive statistics.
Answer these questions: (1) Are you surprised that the bars
for arithmetic and L1 means are so dissimilar, while the L1 means
314 (§6.34)
Task (continued)
and variances are closer? (2) Are you surprised that the variances
for the target and non-target are roughly equal? What does that tell
you about the information represented in the attention matrices?

In [ ]:

# just one layer for now
layeri = 6

# split into separate matrices
Q,K,V = torch.split(activations[f'attn_{layeri}_qkv'],n_emb,dim=-1)
Q.shape,K.shape,V.shape


In [ ]:

# target: means of means and variances
qMeans_t = torch.tensor([ torch.mean(Q[:,-1,:],dim=-1).mean(),
                          torch.mean(abs(Q[:,-1,:]),dim=-1).mean(),
                          torch.var(Q[:,-1,:],dim=-1).mean() ])

# non-target: means of means and variances
qMeans_n = torch.tensor([ torch.mean(Q[:,-2,:],dim=-1).mean(),
                          torch.mean(abs(Q[:,-2,:]),dim=-1).mean(),
                          torch.var(Q[:,-2,:],dim=-1).mean() ])


plt.figure(figsize=(8,3))
plt.bar(np.arange(3)-.2,qMeans_t,width=.5,edgecolor='k',label='Target')
plt.bar(np.arange(3)+.2,qMeans_n,width=.5,edgecolor='k',label='Non-target')
plt.legend()
plt.gca().set(xticks=range(3),xticklabels=['Arithmetic mean','L1 mean','Variance'],ylabel='Descriptive values')

plt.tight_layout()
plt.show()

## Task_2:
- Calculate and visualize six histograms, corresponding to the activa
tion values of the three attention matrices for the target and non
target tokens (Figure 6.14). Use bin edges of-7 to 7 in 234 steps for
all histograms.